## Select residues within certain distance from the substrate in pymol

In [1]:
''' find residues within certain distance from the substrate'''
select near_residues_20A, byres (protein and name CA within 20 of (substrate and name P))

# select protein and ligand and name them as protein and substrate 
# protein and name CA: selects only the alpha-carbon atoms from the protein
# substrate and name P: selects only the phosphate atoms from the substrate (nucleic acids)
# within 18 of Finds atom within 18 A of each other 
# byres: ensures that entire resiudes are selected if any of their atoms meet the criteria 

SyntaxError: invalid syntax (4136223627.py, line 2)

In [2]:
''' save residues position indices in a text file '''
# Step 1: Initialize an empty list
residue_indices_20A = []

# Step 2: Collect residue indices from the selection
cmd.iterate("near_residues_20A", "residue_indices_20A.append(resi)")

# Step 3: Remove duplicates and sort the list
residue_indices_20A = sorted(set(residue_indices_20A), key=lambda x: int(x))

# Step 4: Define the file path
file_path = '/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/MMLV_Termination_20A_residue_indices.txt'

# Step 5: Save the list to a text file
with open(file_path, 'w') as outfile: outfile.write(', '.join(residue_indices_20A))

NameError: name 'cmd' is not defined

## Convert the active site residue indices in the substrate-bound homolog to those in target protein

In [3]:
from Bio import AlignIO

In [4]:
'''import the residues text file generated from pymol'''
# Step 1: Define the file path
file_path = '/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/18A_residue_indices.txt'

# Step 2: Read the contents of the file
with open(file_path, 'r') as infile:
    data = infile.read()

# Step 3: Convert the string data to a list
residue_indices = data.split(', ')

# Step 4: (Optional) Convert residue indices to integers
residue_indices = [int(res) for res in residue_indices]

In [5]:
# MMLV_deltaRNaseH_substrate_bound has some wired index annotations
residue_indices_adjusted =[]
for r in residue_indices:
    if r <449:
        r2=r-23
    if r>449:
        r2 = r-23-6
    residue_indices_adjusted.append(r2)

In [6]:
'''if use a homolog structure, the codes below can convert the homolog residue positions to your target protein positions'''
alignment = AlignIO.read('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/Alignment_MMLV_deltaRNaseH_PE6d.fasta', "fasta")

# define the function to map the homolog residues to the target protein residues by sequence alignment
def index_convert(resList):

  outList = []
  outList_num = []
  res_counter_A = 0
  res_counter_B = 0
  for pos in range(alignment.get_alignment_length()):

    if alignment[1][pos] != '-':
      res_counter_B += 1

    if alignment[0][pos] != '-':
      res_counter_A += 1
      if res_counter_A in resList:
        outList.append(alignment[1][res_counter_B])
        outList_num.append(res_counter_B)
    
  return(outList,outList_num)

In [7]:
residues, num = index_convert(residue_indices_adjusted)
PDB_fix_Res = num
file_path = '/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PDB_fix_Res.txt'
with open(file_path, 'w') as outfile:
    for item in PDB_fix_Res:
        outfile.write(str(item) + '\n')

## Compare fixed residues between AF3 and experimental homolog structure 

In [18]:
import re

# function to read the text file 
def read_file(file_path):
    with open(file_path, 'r') as infile:
        data = infile.read()
        # Split on commas and newlines, strip whitespace, and filter out empty strings
        data_list = [item.strip() for item in re.split(r',\s*|\n', data) if item.strip()]
        # Convert the list items to integers
        data_list = [int(item) for item in data_list]
    return data_list

# read the text file
AF3_18A = read_file('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6d_AF3_18A_residue_indices.txt')
MMLV_substrate_bound_termination_18A = read_file('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/MMLV_Termination_18A_residue_indices.txt')
MMLV_substrate_bound_initiation_18A = read_file('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/MMLV_Initiation_18A_residue_indices.txt')
MMLV_substrate_bound_elongation_18A = read_file('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/MMLV_Elongation_18A_residue_indices.txt')

In [ ]:
# Convert the lists to sets for efficient comparison
AF3_set = set(AF3_18A)
MMLV_substrate_bound_termination_set = set(MMLV_substrate_bound_termination_18A)
MMLV_substrate_bound_initiation_set = set(MMLV_substrate_bound_initiation_18A)
MMLV_substrate_bound_elongation_set = set(MMLV_substrate_bound_elongation_18A)

def compare_indices(set1, set2, set1_name='Set 1', set2_name='Set 2'):
    """
    Compares two sets of indices and prints the common and unique indices.

    Parameters:
    - set1: The first set of indices.
    - set2: The second set of indices.
    - set1_name: A string representing the name of the first set (for display purposes).
    - set2_name: A string representing the name of the second set (for display purposes).

    Returns:
    A dictionary containing:
    - 'common_indices': Indices present in both sets.
    - 'unique_to_set1': Indices unique to the first set.
    - 'unique_to_set2': Indices unique to the second set.
    """
    # Find indices that appear in both sets (common indices)
    common_indices = set1 & set2  # Intersection of the two sets

    # Find indices unique to each set
    unique_to_set1 = set1 - set2   # Indices in set1 but not in set2
    unique_to_set2 = set2 - set1   # Indices in set2 but not in set1

    # Output the results
    print(f"Indices present in both {set1_name} and {set2_name}:")
    print(sorted(common_indices))

    print(f"\nIndices unique to {set1_name}:")
    print(sorted(unique_to_set1))

    print(f"\nIndices unique to {set2_name}:")
    print(sorted(unique_to_set2))

    # Return the results in a dictionary
    return {
        'common_indices': common_indices,
        'unique_to_set1': unique_to_set1,
        'unique_to_set2': unique_to_set2
    }

In [20]:
compare_termination_initiation = compare_indices(MMLV_substrate_bound_termination_set, MMLV_substrate_bound_initiation_set, 'MMLV Termination', 'MMLV Initiation')

Indices present in both MMLV Termination and MMLV Initiation:
[50, 51, 52, 53, 59, 60, 61, 62, 63, 64, 65, 66, 67, 70, 71, 74, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 167, 168, 169, 170, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 203, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 340, 343, 344, 347, 348, 351, 354, 356, 357, 360, 368, 370, 371, 372, 373,

In [21]:
compare_termination_elongation = compare_indices(MMLV_substrate_bound_termination_set, MMLV_substrate_bound_elongation_set, 'MMLV Termination', 'MMLV Elongation')

Indices present in both MMLV Termination and MMLV Elongation:
[33, 34, 40, 50, 51, 52, 53, 59, 60, 61, 62, 63, 64, 65, 66, 67, 70, 71, 74, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 139, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 167, 168, 169, 170, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 265, 266, 267, 268, 269, 270, 271, 272, 273, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 344, 347, 348, 351, 354, 357, 372, 374, 375, 376, 3

In [22]:
compare_termination_AF3 = compare_indices(MMLV_substrate_bound_termination_set, AF3_set, 'MMLV Termination', 'AF3')

Indices present in both MMLV Termination and AF3:
[32, 33, 34, 35, 38, 39, 40, 41, 50, 51, 52, 53, 59, 60, 61, 62, 63, 64, 65, 66, 70, 74, 77, 81, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 167, 168, 169, 170, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 247, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 265, 266, 267, 268, 269, 270, 271, 272, 273, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 340, 343, 344, 347, 348, 351,

## Generate final fixed residues by distance from combination of experiment structure and AF3 structure 

In [36]:
import re

# function to read the text file 
def read_file(file_path):
    with open(file_path, 'r') as infile:
        data = infile.read()
        # Split on commas and newlines, strip whitespace, and filter out empty strings
        data_list = [item.strip() for item in re.split(r',\s*|\n', data) if item.strip()]
        # Convert the list items to integers
        data_list = [int(item) for item in data_list]
    return data_list

# read the text file
AF3_18A = read_file('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6d_AF3_18A_residue_indices.txt')
MMLV_substrate_bound_termination_18A = read_file('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/MMLV_Termination_18A_residue_indices.txt')
AF3_15A = read_file('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6d_AF3_15A_residue_indices.txt')
MMLV_substrate_bound_termination_15A = read_file('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/MMLV_Termination_15A_residue_indices.txt')
AF3_20A = read_file('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6d_AF3_20A_residue_indices.txt')
MMLV_substrate_bound_termination_20A = read_file('/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/MMLV_Termination_20A_residue_indices.txt')    

In [37]:
# Convert the lists to sets for efficient comparison
AF3_18A_set = set(AF3_18A)
MMLV_substrate_bound_termination_18A_set = set(MMLV_substrate_bound_termination_18A)
AF3_15A_set = set(AF3_15A)
MMLV_substrate_bound_termination_15A_set = set(MMLV_substrate_bound_termination_15A)
AF3_20A_set = set(AF3_20A)
MMLV_substrate_bound_termination_20A_set = set(MMLV_substrate_bound_termination_20A)


In [38]:
def fixed_residues_distance(experiment, AF3, output_file):
    
    fixed_residues = experiment 

    # Find indices unique to each set
    unique_to_AF3= AF3 - experiment   # Indices in AF3 but not in experiment

    for index in unique_to_AF3:
        if index < 24 or 449 <= index <=454 or 483 <= index <= 497:
            fixed_residues.add(index)

    with open(output_file, 'w') as file:
        for residue in sorted(fixed_residues):  # Sort for readability
            file.write(f"{residue}\n")

    return fixed_residues

In [39]:
PE6d_fixed_residues_18A_output_file = '/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6d_fixed_residues_18A.txt'
PE6d_fixed_residues_18A = fixed_residues_distance(MMLV_substrate_bound_termination_18A_set, AF3_18A_set, PE6d_fixed_residues_18A_output_file)
PE6d_fixed_residues_18A

{1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 32,
 33,
 34,
 35,
 38,
 39,
 40,
 41,
 50,
 51,
 52,
 53,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 70,
 71,
 73,
 74,
 77,
 81,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 139,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 158,
 167,
 168,
 169,
 170,
 187,
 188,
 189,
 190,
 191,
 192,
 193,
 194,
 195,
 196,
 197,
 198,
 199,
 200,
 201,
 202,
 203,
 204,
 219,
 220,
 221,
 222,
 223,
 224,
 225,
 226,
 227,
 228,
 229,
 247,
 252,
 253,
 254,
 255,
 256,
 257,
 258,
 259,
 260,
 261,
 263,
 265,
 266,
 267,
 268,
 269,
 270,
 271,
 272,
 273,
 274,
 277,
 278,
 279,
 280,
 281,
 282,
 283,
 284,
 285,
 286,
 287,
 288,
 289,
 290,
 291,
 292,
 293,
 294,
 295,
 2

In [40]:
PE6d_fixed_residues_15A_output_file = '/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6d_fixed_residues_15A.txt'
PE6d_fixed_residues_15A = fixed_residues_distance(MMLV_substrate_bound_termination_15A_set, AF3_15A_set, PE6d_fixed_residues_15A_output_file)
PE6d_fixed_residues_15A

{1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 34,
 61,
 63,
 64,
 65,
 66,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 168,
 188,
 189,
 190,
 191,
 192,
 193,
 194,
 195,
 196,
 197,
 198,
 199,
 200,
 201,
 220,
 221,
 222,
 223,
 224,
 225,
 226,
 227,
 252,
 253,
 254,
 255,
 256,
 257,
 258,
 259,
 260,
 266,
 267,
 268,
 269,
 270,
 271,
 272,
 273,
 278,
 279,
 280,
 281,
 282,
 283,
 284,
 285,
 286,
 287,
 288,
 289,
 290,
 291,
 294,
 295,
 296,
 297,
 298,
 299,
 300,
 301,
 302,
 303,
 304,
 305,
 306,
 307,
 308,
 309,
 310,
 311,
 312,
 313,
 314,
 315,
 316,
 317,
 318,
 319,
 320,
 321,
 322,
 323,
 324,
 325,
 326,
 327,
 328,
 329,
 330,
 331,
 332,
 334,
 375,
 376,
 377,
 378,
 379,
 380,
 393,
 394,

In [42]:
PE6d_fixed_residues_20A_output_file = '/Users/taoallen/Desktop/ProteinMPNN_tutorial/PE_RT/Step_1_distance_analysis/PE6d_fixed_residues_20A.txt'
PE6d_fixed_residues_20A = fixed_residues_distance(MMLV_substrate_bound_termination_20A_set, AF3_20A_set, PE6d_fixed_residues_20A_output_file)
PE6d_fixed_residues_20A

{1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 21,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 48,
 49,
 50,
 51,
 52,
 53,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 73,
 74,
 76,
 77,
 78,
 80,
 81,
 86,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 139,
 145,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 158,
 159,
 160,
 164,
 165,
 166,
 167,
 168,
 169,
 170,
 171,
 184,
 185,
 187,
 188,
 189,
 190,
 191,
 192,
 193,
 194,
 195,
 196,
 197,
 198,
 199,
 200,
 201,
 202,
 203,
 204,
 206,
 207,
 218,
 219,
 220,
 221,
 222,
 223,
 224,
 225,
 226,
 227,
 228,
 229,
 243,
 244,
 247,
 248,
 250,
 251,
 252,
 253,
 254,
 255,
 256,
 257,
 258,
 259,
 260,
 261,
